# 02 — DistilBERT Fine-tuning & Evaluation

**Project:** S8 Integrated Project — Multi-Agent Product Review Intelligence  
**Authors:** Hamza Ziouine, Mohamed Nacir, Nour ElHouda Taroujena  
**Date:** 2026-05-02  

This notebook is the narrative version of `src/models/train.py` and `src/models/evaluate.py`.  
It is used as evidence in the report (Section 3 — DL Model, Section 8 — Implementation).  

**Execution order:**
1. Run all cells top to bottom on a machine with a CUDA GPU (RTX 3060 6GB or better).
2. Training takes approximately 25–40 minutes for 4 epochs on an RTX 3060.
3. Evaluation artifacts are written to `report/figures/` and `docs/evaluation.md`.

**DO NOT re-run training if a checkpoint already exists** — use the saved model and jump to the Evaluation section.

## 1. Environment & Reproducibility

In [ ]:
import sys, platform, subprocess, random
from pathlib import Path

import numpy as np
import torch

print(f"Python      : {sys.version}")
print(f"Platform    : {platform.platform()}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Freeze pip environment for reproducibility record
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True)
print("\n--- pip freeze (first 15 lines) ---")
print('\n'.join(freeze.stdout.splitlines()[:15]))

In [ ]:
# Reproducibility seeds — must be set before any model/data operations
SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Seeds set to {SEED}")

## 2. Data Loading

Dataset statistics (from `data/processed/dataset_stats.md`):

| Split | Rows | NEGATIVE (0) | NEUTRAL (1) | POSITIVE (2) |
|-------|------|-------------|------------|-------------|
| train | 45,000 | 15,000 | 15,000 | 15,000 |
| val   | 13,306 | 4,740  | 2,566  | 6,000  |
| test  | 13,306 | 4,741  | 2,565  | 6,000  |

Label mapping: stars 1-2 → NEGATIVE, star 3 → NEUTRAL, stars 4-5 → POSITIVE.

In [ ]:
import pandas as pd
from datasets import Dataset

# Add repo root to path so config.settings resolves correctly
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config.settings import DATA_PROCESSED, SENTIMENT_MODEL_PATH

train_df = pd.read_csv(DATA_PROCESSED / 'train.csv')
val_df   = pd.read_csv(DATA_PROCESSED / 'val.csv')
test_df  = pd.read_csv(DATA_PROCESSED / 'test.csv')

print(f"Train : {len(train_df):,} rows | labels: {dict(train_df['label'].value_counts().sort_index())}")
print(f"Val   : {len(val_df):,} rows  | labels: {dict(val_df['label'].value_counts().sort_index())}")
print(f"Test  : {len(test_df):,} rows  | labels: {dict(test_df['label'].value_counts().sort_index())}")
print(f"\nColumns: {list(train_df.columns)}")
train_df.head(3)

## 3. Model & Training Configuration

**Choice: `distilbert-base-uncased`**
- 60% smaller than BERT-base (66M vs 110M parameters)
- 97% of BERT's NLP task performance (Sanh et al., 2019)
- Inference ~2× faster — critical for production tool calls inside the CrewAI pipeline

**GPU constraint:** RTX 3060 6GB VRAM. With `fp16=True` and `batch_size=16`, peak VRAM usage is ~4.5 GB.  
`gradient_accumulation_steps=2` gives an effective batch size of 32 without doubling memory.

In [ ]:
# Training hyperparameters
CONFIG = {
    'model_name': 'distilbert-base-uncased',
    'num_labels': 3,
    'max_length': 256,
    'batch_size': 16,
    'gradient_accumulation_steps': 2,   # effective batch = 32
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'num_epochs': 4,
    'warmup_ratio': 0.1,
    'early_stopping_patience': 2,
    'fp16': torch.cuda.is_available(),
    'seed': SEED,
}

ID2LABEL = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

print("Training config:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 4. Tokenization

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

def build_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))
    def tokenize(batch):
        return tokenizer(
            batch['text'],
            truncation=True,
            max_length=CONFIG['max_length'],
            padding=False,
        )
    ds = ds.map(tokenize, batched=True, remove_columns=['text'])
    ds = ds.rename_column('label', 'labels')
    ds.set_format(type='torch')
    return ds

train_ds = build_hf_dataset(train_df)
val_ds   = build_hf_dataset(val_df)
test_ds  = build_hf_dataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Train dataset: {len(train_ds)} samples")
print(f"Val dataset  : {len(val_ds)} samples")
print(f"Test dataset : {len(test_ds)} samples")
print(f"Sample token count: {len(train_ds[0]['input_ids'])} (before padding)")

## 5. Training

We use HuggingFace `Trainer` for simplicity and reproducibility. Key choices:
- `eval_strategy='epoch'` — evaluate after every epoch, not every N steps
- `load_best_model_at_end=True` — automatically restores the best checkpoint on val loss
- `EarlyStoppingCallback(patience=2)` — stops if val loss does not improve for 2 consecutive epochs
- `report_to='none'` — no wandb/tensorboard; metrics written to `training_log.json`

In [ ]:
import sklearn.metrics as skm
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = skm.accuracy_score(labels, preds)
    macro_f1 = skm.f1_score(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'macro_f1': macro_f1}

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=CONFIG['num_labels'],
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

training_args = TrainingArguments(
    output_dir=str(SENTIMENT_MODEL_PATH),
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'] * 2,
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    warmup_ratio=CONFIG['warmup_ratio'],
    fp16=CONFIG['fp16'],
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=1,
    seed=CONFIG['seed'],
    data_seed=CONFIG['seed'],
    report_to='none',
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG['early_stopping_patience'])],
)

print("Model loaded. Starting training...")
train_result = trainer.train()
print("Training complete.")
print(train_result.metrics)

In [ ]:
import json, time

# Save best model + tokenizer
trainer.save_model(str(SENTIMENT_MODEL_PATH))
tokenizer.save_pretrained(str(SENTIMENT_MODEL_PATH))

# Build and save training log
history = [
    {
        'epoch': e.get('epoch'),
        'eval_loss': e.get('eval_loss'),
        'eval_accuracy': e.get('eval_accuracy'),
        'eval_macro_f1': e.get('eval_macro_f1'),
    }
    for e in trainer.state.log_history if 'eval_loss' in e
]

training_log = {
    **CONFIG,
    'effective_batch_size': CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps'],
    'train_samples': len(train_ds),
    'val_samples': len(val_ds),
    'best_checkpoint': str(SENTIMENT_MODEL_PATH),
    'per_epoch_metrics': history,
}

log_path = SENTIMENT_MODEL_PATH / 'training_log.json'
with open(log_path, 'w', encoding='utf-8') as fh:
    json.dump(training_log, fh, indent=2)

print(f"Model saved to: {SENTIMENT_MODEL_PATH}")
print(f"Training log  : {log_path}")
print("\nPer-epoch validation metrics:")
for h in history:
    print(f"  Epoch {h['epoch']:.0f} | loss={h['eval_loss']:.4f} | acc={h['eval_accuracy']:.4f} | macro_f1={h['eval_macro_f1']:.4f}")

## 6. Training Curves

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIGURES_DIR = REPO_ROOT / 'report' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

epochs = [h['epoch'] for h in history]
eval_losses = [h['eval_loss'] for h in history]
eval_accs   = [h['eval_accuracy'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epochs, eval_losses, marker='o', color='steelblue', label='Val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Validation Loss per Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, eval_accs, marker='o', color='seagreen', label='Val accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy per Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.suptitle('DistilBERT Fine-tuning — Training Curves', fontsize=13)
fig.tight_layout()
out = FIGURES_DIR / 'training_curves.png'
fig.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

## 7. Evaluation on Test Set

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support,
)

LABEL_NAMES = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load best saved model
eval_tokenizer = AutoTokenizer.from_pretrained(str(SENTIMENT_MODEL_PATH))
eval_model = AutoModelForSequenceClassification.from_pretrained(str(SENTIMENT_MODEL_PATH))
eval_model.to(DEVICE).eval()

texts = test_df['text'].tolist()
true_labels = test_df['label'].to_numpy()

# Batched inference
BATCH = 64
all_preds, all_probs = [], []
for start in range(0, len(texts), BATCH):
    batch = texts[start:start + BATCH]
    enc = eval_tokenizer(batch, truncation=True, max_length=256, padding=True, return_tensors='pt')
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = eval_model(**enc).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    all_preds.append(np.argmax(probs, axis=-1))
    all_probs.append(probs)

pred_labels = np.concatenate(all_preds)
all_probs   = np.concatenate(all_probs)

overall_acc = accuracy_score(true_labels, pred_labels)
macro_f1    = f1_score(true_labels, pred_labels, average='macro', zero_division=0)
precisions, recalls, f1s, _ = precision_recall_fscore_support(
    true_labels, pred_labels, labels=[0, 1, 2], zero_division=0
)

print(f"Test Accuracy : {overall_acc:.4f}")
print(f"Test Macro F1 : {macro_f1:.4f}")
print()
print(classification_report(true_labels, pred_labels, target_names=LABEL_NAMES, zero_division=0))

## 8. Confusion Matrix

In [ ]:
import seaborn as sns

cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix — Test Set (DistilBERT fine-tuned)')
fig.tight_layout()
out = FIGURES_DIR / 'confusion_matrix.png'
fig.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

## 9. Per-class Metrics & Baselines

In [ ]:
# Per-class bar chart
x = np.arange(len(LABEL_NAMES))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width, precisions, width, label='Precision', color='steelblue')
ax.bar(x,         recalls,    width, label='Recall',    color='seagreen')
ax.bar(x + width, f1s,        width, label='F1',        color='coral')
ax.set_xticks(x); ax.set_xticklabels(LABEL_NAMES)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Per-class Precision / Recall / F1 — Test Set')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / 'per_class_metrics.png'
fig.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

# Baselines
rng = np.random.default_rng(42)
random_preds = rng.integers(0, 3, size=len(true_labels))
majority_preds = np.full(len(true_labels), np.bincount(true_labels).argmax())

print("\n--- Baseline comparison ---")
print(f"Random classifier  accuracy: {accuracy_score(true_labels, random_preds):.4f}")
print(f"Majority class     accuracy: {accuracy_score(true_labels, majority_preds):.4f}")
print(f"DistilBERT (ours)  accuracy: {overall_acc:.4f}")

## 10. Qualitative Examples

In [ ]:
import textwrap

ID2LABEL_NB = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}

def show_examples(texts, true_labels, pred_labels, probs, correct=True, n=5):
    mask = (true_labels == pred_labels) if correct else (true_labels != pred_labels)
    indices = np.where(mask)[0]
    rng = np.random.default_rng(42)
    chosen = rng.choice(indices, size=min(n, len(indices)), replace=False)
    label_str = 'CORRECT' if correct else 'MISCLASSIFIED'
    print(f"\n=== {n} {label_str} examples ===")
    for i, idx in enumerate(sorted(chosen), 1):
        print(f"\n[{i}] True: {ID2LABEL_NB[int(true_labels[idx])]:10s}  "
              f"Predicted: {ID2LABEL_NB[int(pred_labels[idx])]:10s}  "
              f"Confidence: {probs[idx].max():.3f}")
        print(f"    Text: {textwrap.shorten(texts[idx], 200)}")

show_examples(texts, true_labels, pred_labels, all_probs, correct=True)
show_examples(texts, true_labels, pred_labels, all_probs, correct=False)

## 11. Write evaluation.md

Calling `src/models/evaluate.py`'s `evaluate_model()` to generate the full markdown artifact used by A11b.

In [ ]:
from src.models.evaluate import evaluate_model

metrics = evaluate_model(model_dir=SENTIMENT_MODEL_PATH, batch_size=64)
print(f"docs/evaluation.md written. Accuracy={metrics['accuracy']:.4f}, Macro F1={metrics['macro_f1']:.4f}")